<a href="https://colab.research.google.com/github/yd659/Accenture-1K-contract-review-challenge/blob/dev/notebooks/Yutika_Deshpande_Task_9_Dataset_Preparation_Tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUAD Dataset Preparation & Tokenization (Task #9)

## 1. Install and import packages

In [15]:
%pip -q install "transformers>=4.45,<5" torch numpy pandas


In [16]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch: 2.11.0+cpu
CUDA available: False


## 2. Upload the processed chunk data

In [17]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving cuad_test_split.jsonl to cuad_test_split (1).jsonl
Uploaded: ['cuad_test_split (1).jsonl']


In [18]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving cuad_train_split.jsonl to cuad_train_split (1).jsonl
Uploaded: ['cuad_train_split (1).jsonl']


In [19]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving cuad_val_split.jsonl to cuad_val_split (1).jsonl
Uploaded: ['cuad_val_split (1).jsonl']


In [20]:
# Expected input filenames
COMBINED_FILE = Path("cuad_all_510_chunked_multilabel.jsonl")
SPLIT_FILES = {
    "train": Path("cuad_train_split.jsonl"),
    "validation": Path("cuad_val_split.jsonl"),
    "test": Path("cuad_test_split.jsonl"),
}

if all(path.exists() for path in SPLIT_FILES.values()):
    frames = {
        split: pd.read_json(path, lines=True)
        for split, path in SPLIT_FILES.items()
    }
    print("Loaded Aditi's existing contract-level split files.")
elif COMBINED_FILE.exists():
    combined_df = pd.read_json(COMBINED_FILE, lines=True)

    # Match Aditi's reproducible 80/10/10 split by contract_id (seed 42).
    contract_ids = combined_df["contract_id"].drop_duplicates().to_numpy()
    rng = np.random.default_rng(42)
    rng.shuffle(contract_ids)

    n_contracts = len(contract_ids)
    n_train = int(n_contracts * 0.80)
    n_validation = int(n_contracts * 0.10)
    train_ids = set(contract_ids[:n_train])
    validation_ids = set(contract_ids[n_train:n_train + n_validation])
    test_ids = set(contract_ids[n_train + n_validation:])

    frames = {
        "train": combined_df[combined_df["contract_id"].isin(train_ids)].copy(),
        "validation": combined_df[combined_df["contract_id"].isin(validation_ids)].copy(),
        "test": combined_df[combined_df["contract_id"].isin(test_ids)].copy(),
    }
    print("Created Aditi-style seeded splits from the combined chunk file.")
else:
    raise FileNotFoundError(
        "Upload cuad_all_510_chunked_multilabel.jsonl or all three split JSONL files."
    )

for name, frame in frames.items():
    print(f"{name}: {len(frame):,} chunks, "
          f"{frame['contract_id'].nunique():,} contracts")


Loaded Aditi's existing contract-level split files.
train: 11,500 chunks, 408 contracts
validation: 1,998 chunks, 51 contracts
test: 1,226 chunks, 51 contracts


## 3. Validate rows and contract-level splits

In [21]:
required_columns = {"contract_id", "chunk_id", "chunk_text", "labels"}
for name, frame in frames.items():
    missing_columns = required_columns - set(frame.columns)
    assert not missing_columns, f"{name} is missing columns: {missing_columns}"
    assert frame["contract_id"].notna().all(), f"{name} has missing contract IDs"
    assert frame["chunk_text"].notna().all(), f"{name} has missing chunk text"
    assert frame["chunk_id"].notna().all(), f"{name} has missing chunk IDs"

contract_sets = {
    name: set(frame["contract_id"].unique())
    for name, frame in frames.items()
}
assert contract_sets["train"].isdisjoint(contract_sets["validation"])
assert contract_sets["train"].isdisjoint(contract_sets["test"])
assert contract_sets["validation"].isdisjoint(contract_sets["test"])

all_contracts = set.union(*contract_sets.values())
assert len(all_contracts) == 510, f"Expected 510 contracts, found {len(all_contracts)}"

print("PASS: 510 contracts, no missing IDs/text, and no contract appears in multiple splits.")


PASS: 510 contracts, no missing IDs/text, and no contract appears in multiple splits.


## 4. Build the fixed 41-label multi-hot vectors

The mapping is derived once from the union of the three splits, sorted for a stable order, and then reused unchanged. The check must find all 41 CUAD categories.

In [22]:
def as_label_list(value):
    # JSONL should load labels as lists; handle nulls defensively.
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    raise TypeError(f"Expected a list of labels, got {type(value).__name__}: {value!r}")

for frame in frames.values():
    frame["labels"] = frame["labels"].apply(as_label_list)

label_names = sorted({
    label
    for frame in frames.values()
    for labels in frame["labels"]
    for label in labels
})
assert len(label_names) == 41, (
    f"Expected 41 unique labels, found {len(label_names)}. "
    "Compare labels with category_descriptions.csv before continuing."
)

label_to_id = {label: index for index, label in enumerate(label_names)}

def encode_multi_hot(labels):
    unknown = set(labels) - set(label_to_id)
    assert not unknown, f"Labels not in mapping: {unknown}"
    vector = [0.0] * len(label_names)
    for label in labels:
        vector[label_to_id[label]] = 1.0
    return vector

for frame in frames.values():
    frame["labels_multi_hot"] = frame["labels"].apply(encode_multi_hot)
    assert frame["labels_multi_hot"].apply(len).eq(41).all()

with open("cuad_label_to_id.json", "w") as f:
    json.dump(label_to_id, f, indent=2)

print("PASS: all splits use the same 41-label order.")
print("First five labels:", label_names[:5])


PASS: all splits use the same 41-label order.
First five labels: ['Affiliate License-Licensee', 'Affiliate License-Licensor', 'Agreement Date', 'Anti-Assignment', 'Audit Rights']


## 5. Tokenize with the DistilRoBERTa tokenizer

Chunk creation used a 512-token limit. This cell uses the same checkpoint name as Michael's architecture notebook and truncates only if needed. Padding happens within each batch.

In [23]:
MODEL_NAME = "distilbert/distilroberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer:", MODEL_NAME)
print("Configured maximum sequence length:", MAX_LENGTH)


Tokenizer: distilbert/distilroberta-base
Configured maximum sequence length: 512


## 6. Create Dataset and DataLoaders

In [24]:
class CUADChunkDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame["chunk_text"].astype(str).tolist()
        self.targets = frame["labels_multi_hot"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return self.texts[index], self.targets[index]


def collate_chunks(rows):
    texts, targets = zip(*rows)
    encoded = tokenizer(
        list(texts),
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor(
        np.asarray(targets, dtype=np.float32),
        dtype=torch.float32,
    )
    return encoded

loaders = {
    name: DataLoader(
        CUADChunkDataset(frame),
        batch_size=BATCH_SIZE,
        shuffle=(name == "train"),
        collate_fn=collate_chunks,
    )
    for name, frame in frames.items()
}

print("Created train, validation, and test DataLoaders.")


Created train, validation, and test DataLoaders.


## 7. Smoke-test the batches and save prepared split files

In [25]:
train_batch = next(iter(loaders["train"]))
print("input_ids shape:", tuple(train_batch["input_ids"].shape))
print("attention_mask shape:", tuple(train_batch["attention_mask"].shape))
print("labels shape:", tuple(train_batch["labels"].shape))

assert train_batch["input_ids"].shape[1] <= MAX_LENGTH
assert train_batch["labels"].shape[1] == 41
assert train_batch["labels"].dtype == torch.float32
print("PASS: tokenized batch is within 512 tokens and has 41 targets per chunk.")

for name, frame in frames.items():
    output_path = f"cuad_{name}_transformer_ready.jsonl"
    frame.to_json(output_path, orient="records", lines=True)
    print(f"Saved {output_path} ({len(frame):,} rows)")


input_ids shape: (8, 512)
attention_mask shape: (8, 512)
labels shape: (8, 41)
PASS: tokenized batch is within 512 tokens and has 41 targets per chunk.
Saved cuad_train_transformer_ready.jsonl (11,500 rows)
Saved cuad_validation_transformer_ready.jsonl (1,998 rows)
Saved cuad_test_transformer_ready.jsonl (1,226 rows)


In [26]:
from google.colab import files
import zipfile

output_files = [
    "cuad_train_transformer_ready.jsonl",
    "cuad_validation_transformer_ready.jsonl",
    "cuad_test_transformer_ready.jsonl",
    "cuad_label_to_id.json",
]
with zipfile.ZipFile("task9_prepared_outputs.zip", "w") as archive:
    for path in output_files:
        archive.write(path)
files.download("task9_prepared_outputs.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
!pip -q install transformers

In [28]:
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

files = {
    "train": "cuad_train_transformer_ready.jsonl",
    "validation": "cuad_validation_transformer_ready.jsonl",
    "test": "cuad_test_transformer_ready.jsonl",
}

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")
data = {}

for split, filename in files.items():
    df = pd.read_json(filename, lines=True)

    assert df["labels_multi_hot"].apply(
        lambda labels: isinstance(labels, list) and len(labels) == 41
    ).all(), f"{split}: found a label vector that isn't length 41"

    encoded = tokenizer(
        df["chunk_text"].fillna("").astype(str).tolist(),
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    df["input_ids"] = encoded["input_ids"]
    df["attention_mask"] = encoded["attention_mask"]

    output = f"cuad_{split}_tokenized.jsonl"
    df.to_json(output, orient="records", lines=True)

    data[split] = df
    print(f"{split}: {len(df)} chunks; saved {output}")

# Confirm tokenized inputs are 512 tokens and contracts don't cross splits.
for split, df in data.items():
    assert df["input_ids"].apply(lambda x: len(x) == 512).all()
    assert df["attention_mask"].apply(lambda x: len(x) == 512).all()

contract_sets = {
    split: set(df["contract_id"])
    for split, df in data.items()
}
assert contract_sets["train"].isdisjoint(contract_sets["validation"])
assert contract_sets["train"].isdisjoint(contract_sets["test"])
assert contract_sets["validation"].isdisjoint(contract_sets["test"])

print("Checks passed: 512-token inputs, 41 labels, and no contract leakage.")

train: 11500 chunks; saved cuad_train_tokenized.jsonl
validation: 1998 chunks; saved cuad_validation_tokenized.jsonl
test: 1226 chunks; saved cuad_test_tokenized.jsonl
Checks passed: 512-token inputs, 41 labels, and no contract leakage.
